# 68. Decoupled BO：Study・Ask-and-Tell・非同期評価

強度と導電率の測定時間が異なる実験を想定します。Studyは候補と観測履歴を管理し、モデルの学習や獲得関数は公開APIを利用します。測定済みの値と未完了の測定を区別することが重要です。

In [ ]:
import torch
import matplotlib.pyplot as plt
from botorch.models import ModelListGP, SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import SumMarginalLogLikelihood
from robotorchan.study import create_study, create_decoupled_proposer
from robotorchan.study.trial import TrialState
from robotorchan.workflow import ProblemSpec, recommend, check_compatibility

torch.manual_seed(0)
torch.set_num_threads(1)
names = ("strength", "conductivity")
grid = torch.linspace(0, 1, 32, dtype=torch.double).unsqueeze(-1)
masks = torch.tensor([[True, False], [False, True], [True, True]])
costs = {"strength": 3., "conductivity": 1.}

def measure(x):
    return {"strength": 1-(x-.75)**2, "conductivity": 1-(x-.25)**2}

def model_factory(data):
    models = [SingleTaskGP(*data[name]) for name in names]
    model = ModelListGP(*models)
    fit_gpytorch_mll(SumMarginalLogLikelihood(model.likelihood, model))
    model.eval()
    return model

def seed_study(study):
    for x in torch.linspace(0, 1, 5).tolist():
        trial = study.add_trial({"x": x}, evaluation_mask=names)
        study.start_trial(trial.number)
        study.tell(trial, observations=measure(x))

## 1. Capability-based Selection

ProblemSpecで複数出力・部分観測・未完了測定への対応を指定します。recommendは互換性のある構成を提案しますが、モデルの学習データや効用までは自動決定しません。

In [ ]:
spec = ProblemSpec(purpose="active_learning", output_type="multi",
                   decoupled=True, masked_pending=True)
recommendations = recommend(spec, model="ModelListGP",
                            acquisition="qDecoupledExpectedPredictiveInformationGain")
assert recommendations
recommendation = recommendations[0]
print(recommendation)
print(check_compatibility(spec, model="HeterogeneousModel",
                          acquisition="qDecoupledParticleKnowledgeGradient",
                          optimizer="optimize_acqf_decoupled"))

## 2. 実際のモデル学習と候補提案をStudyに接続する

model_factoryは出力名ごとの観測Tensorを受け取ります。初期観測は両出力5点、費用20です。総予算40には初期費用も含まれます。EPIGは未完了測定の情報を条件として次の候補を評価できます。

In [ ]:
proposer = create_decoupled_proposer(
    parameter_names=("x",), output_names=names, evaluation_masks=masks,
    model_factory=model_factory, recommendation=recommendation, problem_spec=spec,
    acquisition_kwargs={"target_X": grid, "X_evaluation_mask": masks[:1],
                        "observation_noise": grid.new_tensor([.01, .01])},
    optimizer_kwargs={"optimizer": "sobol", "optimizer_options": {"num_samples": 8}},
    pending_fantasies=True, output_costs=costs, total_budget=40.,
)
kwargs = dict(search_space={"x": (0., 1.)}, output_names=names,
              decoupled_proposer=proposer)
study = create_study(**kwargs)
seed_study(study)
trial = study.ask()
print(trial.number, trial.params, trial.evaluation_mask)
values = measure(trial.params["x"])
study.tell(trial, observations={name: values[name] for name in trial.evaluation_mask})
print(study.get_trial(trial.number).state)

## 3. 部分tellと非同期ask

二項目を測定する実験を登録し、先に強度だけが返る状況を再現します。導電率は未完了のまま、別の候補を実際のEPIGで提案します。すでに観測した強度は学習データに入り、未完了の導電率だけがpendingになります。

In [ ]:
slow = study.add_trial({"x": .45}, evaluation_mask=names)
study.start_trial(slow.number)
partial = study.tell(slow, observations={"strength": measure(.45)["strength"]})
assert partial.state is TrialState.RUNNING
print(partial.observations, partial.evaluation_mask)
fast = study.ask()
print("Next:", fast.number, fast.params, fast.evaluation_mask)
values = measure(fast.params["x"])
study.tell(fast, observations={name: values[name] for name in fast.evaluation_mask})
done = study.tell(slow, observations={"conductivity": measure(.45)["conductivity"]})
assert done.state is TrialState.COMPLETE
print(done.observations)

## 4. 失敗・再試行・実測費用

測定の一部が失敗しても取得済みの値は失われません。失敗した測定にも費用が発生する場合はactual_costsに記録します。retry_failedは未取得項目だけを同じ設計点で再試行し、残予算を確認します。

In [ ]:
failed = study.add_trial({"x": .65}, evaluation_mask=names)
study.start_trial(failed.number)
study.tell(failed, observations={"strength": measure(.65)["strength"]},
           actual_costs={"strength": 3., "conductivity": 1.}, state=TrialState.FAIL)
retry = study.retry_failed(failed.number)
assert retry.params == failed.params
assert retry.evaluation_mask == ("conductivity",)
study.tell(retry, observations={"conductivity": measure(.65)["conductivity"]})
print(study.get_trial(failed.number).observations, retry.evaluation_mask)

## 5. 履歴から再開する

initial_trialsに既存の履歴を渡してStudyを再構成します。これはメモリ上の履歴復元であり、外部データベースへの永続化ではありません。モデルは観測履歴から再学習されます。

In [ ]:
restored = create_study(**kwargs, initial_trials=study.trials)
assert len(restored.trials) == len(study.trials)
frame = restored.trials_dataframe()
display(frame)
next_trial = restored.ask()
print(next_trial.number, next_trial.params, next_trial.evaluation_mask)
values = measure(next_trial.params["x"])
restored.tell(next_trial, observations={name: values[name] for name in next_trial.evaluation_mask})

## 6. 観測履歴の可視化

出力ごとの観測点と累積実測費用を描きます。失敗した測定の実測費用も含め、観測値のない項目を学習に追加しません。ここでは完了・失敗済み履歴のみを集計します。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for name in names:
    observed = [t for t in restored.trials if name in t.observations]
    axes[0].scatter([t.params["x"] for t in observed],
                    [t.observations[name] for t in observed], label=name)
axes[0].set_xlabel("Design")
axes[0].set_ylabel("Observation")
axes[0].legend()
spent = 0.
history = [spent]
for t in restored.trials:
    spent += sum(t.actual_costs.get(name, costs[name] if name in t.observations else 0.)
                 for name in names)
    history.append(spent)
axes[1].step(range(len(history)), history, where="post")
axes[1].axhline(40., linestyle="--", label="Budget")
axes[1].set_xlabel("Trial history index")
axes[1].set_ylabel("Cumulative cost")
axes[1].legend()
plt.show()
assert spent <= 40.

## まとめ

Studyは候補提案・観測・失敗・再開を一つの履歴にまとめます。未完了測定の扱いは獲得関数の対応能力に依存します。Particle KGや通常の部分観測KGにはmasked pendingを渡せません。実験装置の並列実行や外部永続化は利用者側の責務です。